# Nemotron-3-Nano LoRA SFT — Tong Hui Kang Style CoT Reproduction

## What This Notebook Does

After [Tong Hui Kang's brilliant 0.85 solution](https://www.kaggle.com/code/huikang/tinker-submission-notebook) was published,
I attempted to reproduce his approach as closely as possible with my own setup.

**Approach**: This notebook follows option 1 — adopt Tong's SFT methodology and recreate his deterministic CoT traces programmatically.

### What I Managed to Reproduce
- **Deterministic CoT generation** for all 7 categories (GC, NC, UC, TE, BM, ET Numeric, Cryptarithm)
- **Integer long division** for GC/UC (instead of float operations)
- **1-100 Roman numeral table** for NC
- **77-word dictionary + char-by-char mapping** for TE
- **354-candidate per-bit testing + stride detection** for BM
- **32 operators × 4 pairings + neg format detection** for ET Numeric
- **Concatenation detection** for Cryptarithm
- **1st-place training hyperparameters**: LR=2e-4, batch=64, 1 epoch, linear decay, no boxed weight, no NEFTune

### What I Could NOT Reproduce
- **Mamba gate_proj + x_proj → in_proj SVD merging** (Tinker-specific)
- **min logprob < -0.69 detection and CoT iteration** (requires Tinker dashboard)
- **8192 Context Length on kaggle GPU** 
- **Perfect Tinker training infrastructure** (8 sec/step vs my 60+ sec/step)

### Important Note
This is a rough reproduction attempt. If you happen to find this notebook helpful and want to upvote it,
**please upvote [Tong's original notebook](https://www.kaggle.com/code/huikang/tinker-submission-notebook) first** — it's the perfect, original work.

### Local Evaluation Results 
(exp026/s005, eval_split 500 samples)
| Category | Accuracy |
|----------|----------|
| GC | 93.8% |
| NC | 100% |
| UC | 98.8% |
| TE | 95.2% |
| ET | 27.2% |
| BM | 50.0% |
| **Total** | **0.768** |

(exp026/s006, eval_split 500 samples)
| Category | Accuracy |
|----------|----------|
| GC | 95.1% |
| NC | 100% |
| UC | 97.5% |
| TE | 97.6% |
| ET | 29.6% |
| BM | 70.2% |
| **Total** | **0.814** |

(exp026/s011, eval_split 500 samples)
| Category | Accuracy |
|----------|----------|
| GC | 95.1% |
| NC | 100% |
| UC | 98.8% |
| TE | 96.4% |
| ET | 27.2% |
| BM | 70.2% |
| **Total** | **0.810** |

(exp026/s012, eval_split 500 samples)
| Category | Accuracy |
|----------|----------|
| GC | 96.3% |
| NC | 100% |
| UC | 100% |
| TE | 96.4% |
| ET | 29.6% |
| BM | 78.7% |
| **Total** | **0.834** |

In [ ]:
# I need to process the data from the six categories derived from train_split.csv.
# train.csv (original 9500 record) →  train_split.csv (9000 record) + eval_split.csv (500 record)
# train_split.csv → gravity_physics.csv + numeral_system.csv + unit_conversion.csv + text_decryption.csv + bit_manipulation_including_wrong.csv + equation_numeric_including_wrong.csv + cryptarithm_including_wrong.csv
# Since I created `eval_split`, I felt there wasn’t quite enough data in the bit manipulation category, so I supplemented it with synthetic data (bit_manipulation_synth_v2.csv, bit_manipulation_synth_including_wrong_v2.csv).

# ○ Three easy categories.
# gravity_physics.csv, numeral_system.csv, unit_conversion.csv

# ○ Slightly easy category.
# text_decryption.csv

# ○ Difficult category
# bit_manipulation.csv                          (accurate CoT only)
# bit_manipulation_including_wrong.csv          (including wrong CoT) 
# bit_manipulation_synth_v2.csv                 (synthetic data, accurate CoT only)    
# bit_manipulation_synth_including_wrong_v2.csv (synthetic data, wrong CoT) 

# ○ Very difficult category 1st
# equation_numeric.csv                       [deduce + guess] (accurate CoT only)
# equation_numeric_including_wrong.csv       [deduce + guess] (including wrong CoT)
# equation_numeric_guess.csv                 [guess only]     (accurate CoT only)
# equation_numeric_guess_including_wrong.csv [guess only]     (including wrong CoT)

# ○ Very difficult category 2nd
# cryptarithm.csv                       [deduce + guess] (accurate CoT only)
# cryptarithm_including_wrong.csv       [deduce + guess] (including wrong CoT)
# cryptarithm_guess.csv                 [guess only]     (accurate CoT only)
# cryptarithm_guess_including_wrong.csv [guess only]     (including wrong CoT)

## Mode Selection

In [ ]:
# ============================================================
# MODE SELECTION — set exactly one to 1
# ============================================================

# if 1: # Mode A: Train LoRA from scratch on Kaggle GPU # Sorry, this option enables us to train the model on up to only 4096 context length. Becaude of OOM on Kaggle GPU.
if 0:  # Mode A: Train LoRA from scratch on Kaggle GPU # Sorry, this option enables us to train the model on up to only 4096 context length. Becaude of OOM on Kaggle GPU.
    TRAIN_ON_KAGGLE = 1 
    USE_PRETRAINED = 0
else: # Mode B: Use pre-trained LoRA weights from dataset and just package them (for saving kaggle GPU resources) 
    TRAIN_ON_KAGGLE = 0
    USE_PRETRAINED = 1

assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, \
    "Set exactly one of TRAIN_ON_KAGGLE / USE_PRETRAINED to 1."


# PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/exp026-s005-lora" # I trained an exp026-s005 setting model locally by 8192 context length
# PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/exp026-s006-lora" # I trained an exp026-s006 setting model locally by 8192 context length
# PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/exp026-s011-lora" # I trained an exp026-s011 setting model locally by 8192 context length and LR 2.4e-4
PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/exp026-s012-lora" # I trained an exp026-s012 setting model locally by 8192 context length and LR 2.4e-4


# ============================================================
# DATA VARIANT (only used when TRAIN_ON_KAGGLE=1)
# ============================================================
# 0 = just training test
# 1 = exp026/s005 like (downsampling ratios)
# 2 = exp026/s006 like (priority upweighting exps026/s005 case)
# 3 = exp026/s011 like (downsampling ratios)
# 4 = exp026/s012 like (priority upweighting exps026/s011 case)

DATA_VARIANT = 0 # test
# DATA_VARIANT = 1
# DATA_VARIANT = 2
# DATA_VARIANT = 3
# DATA_VARIANT = 4

## Setup & Model Loading

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "--target", target,
        "--upgrade",
        "--ignore-installed",
        wheel,
    ],
    check=True,
)
site.addsitedir(target)
print("Triton installed to", target)

In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC)
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst
        print(f"ptxas-blackwell copied to {ptxas_dst}")

    import importlib, triton
    importlib.reload(triton)
    print("Triton:", triton.__version__)

In [ ]:
# Install trl if needed (for training mode)
if TRAIN_ON_KAGGLE:
    try:
        import trl
        print(f"trl already installed: {trl.__version__}")
    except ImportError:
        offline_path = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/"
        if os.path.exists(offline_path):
            subprocess.run(f"pip install --no-index --find-links={offline_path} trl", shell=True)
        else:
            subprocess.run("pip install trl", shell=True)
        import trl
        print(f"trl installed: {trl.__version__}")

In [ ]:
if TRAIN_ON_KAGGLE:
    import glob, importlib.util, os, subprocess, sys, types

    def sh(cmd: str, check: bool = True):
        print("+", cmd)
        return subprocess.run(cmd, shell=True, check=check)

    def find_spec(name: str) -> bool:
        return importlib.util.find_spec(name) is not None

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    all_mamba = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal_conv1d-*.whl")
    print("mamba wheels:", all_mamba)
    print("causal_conv1d wheels:", all_causal)

    if not find_spec("causal_conv1d") and all_causal:
        sh(f"pip install --no-deps {all_causal[-1]}")
    if not find_spec("mamba_ssm") and all_mamba:
        sh(f"pip install --no-deps {all_mamba[-1]}")

    import mamba_ssm
    print("mamba_ssm:", mamba_ssm.__version__)

In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import kagglehub
    import logging, warnings

    # Suppress verbose loading output
    logging.getLogger("transformers").setLevel(logging.ERROR)
    warnings.filterwarnings("ignore", message=".*torch_dtype.*")
    
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, torch_dtype=torch.bfloat16, trust_remote_code=True,
        attn_implementation="eager")
    print(f"Model loaded: {type(model).__name__}")

In [ ]:
if TRAIN_ON_KAGGLE:
    from peft import LoraConfig, get_peft_model, TaskType

    # 1st place style: 9 targets, r=32, alpha=32, no dropout
    lora_config = LoraConfig(
        r=32,
        lora_alpha=32,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "in_proj","out_proj","up_proj","down_proj","lm_head"],
        lora_dropout=0.0,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
else:
    print("USE_PRETRAINED=1: skipping PEFT model setup")

## Mode A: Train on Kaggle

In [ ]:
if TRAIN_ON_KAGGLE:
    import pandas as pd
    import re
    import random
    from datasets import Dataset as HFDataset
    from trl import SFTTrainer, SFTConfig

    SEED = 123
    PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

    # --- Load CoT data from uploaded dataset ---
    TONG_DIR = "/kaggle/input/datasets/konbu17/exp024-tong-style-cot-data/type_tong"

    # Data variant selection
    if DATA_VARIANT == 0:
        # for training test
        SAMPLES = {
            "numeral_system.csv": 10,
            "gravity_physics.csv": 10,
            "unit_conversion.csv": 10,
            "text_decryption.csv": 10,
            "bit_manipulation.csv": 10,
            "bit_manipulation_synth_including_wrong_v2.csv": 10,
            "equation_numeric.csv": 10,
            "cryptarithm.csv": 10,
        }
        print("Data Variant 3: Just for training test")
    
    elif DATA_VARIANT == 1:
        # exp026/s005 like — downsampling ratios
        SAMPLES = {
            "numeral_system.csv": 600,
            "gravity_physics.csv": 1000,
            "unit_conversion.csv": 1000,
            "text_decryption.csv": 1492,
            "bit_manipulation_including_wrong.csv": 1508,
            "equation_numeric.csv": 535,
            "cryptarithm.csv": 69,
        }
        print("Data Variant 1: downsampling")
        
    elif DATA_VARIANT == 2:
        # exp026/s006 like — priority upweighting exp026/s005 case
        BASE_SAMPLES = {
            "numeral_system.csv": 600,
            "gravity_physics.csv": 1000,
            "unit_conversion.csv": 1000,
            "text_decryption.csv": 1492,
            "bit_manipulation_including_wrong.csv": 1508,
            "equation_numeric.csv": 535,
            "cryptarithm.csv": 69,
        }
        # Load priority IDs from uploaded dataset
        PRIORITY_FILE = "/kaggle/input/datasets/konbu17/exp024-tong-style-cot-data/priority/exp026_s005_priority.txt"
        priority_ids = set()
        if os.path.exists(PRIORITY_FILE):
            with open(PRIORITY_FILE) as pf:
                priority_ids = {line.strip() for line in pf if line.strip()}
            print(f"Loaded {len(priority_ids)} priority problem IDs")
        else:
            print(f"WARNING: Priority file not found: {PRIORITY_FILE}")

        print("Data Variant 2: Priority upweighting (exp026/s005 style)")

        # Build records with ID tracking for priority duplication
        records = []
        id_records = []
        for fname, n in BASE_SAMPLES.items():
            path = os.path.join(TONG_DIR, fname)
            df = pd.read_csv(path)
            df = df[df["generated_cot"].notna() & (df["generated_cot"].str.len() > 5)]
            sampled = df.sample(n=min(n, len(df)), random_state=SEED)
            print(f"  {fname}: {len(sampled)}/{n}")
            for _, row in sampled.iterrows():
                rec = build_record(row["prompt"], row["answer"], row["generated_cot"])
                records.append(rec)
                id_records.append((str(row["id"]), rec))

        print(f"Base records: {len(records)}")

        # Add priority duplicates (hard problems get 2x training weight)
        dup_count = 0
        for pid, rec in list(id_records):
            if pid in priority_ids:
                records.append(dict(rec))
                dup_count += 1
        print(f"Priority duplicates: {dup_count}")
        print(f"Total records: {len(records)}")

    elif DATA_VARIANT == 3:
        # exp026/s011 like — downsampling ratios
        SAMPLES = {
            "numeral_system.csv": 600,
            "gravity_physics.csv": 1200,
            "unit_conversion.csv": 1150,
            "text_decryption.csv": 1492,
            "bit_manipulation_including_wrong.csv": 1508,
            "bit_manipulation_synth_including_wrong_v2.csv": 500,
            "equation_numeric.csv": 535,
            "cryptarithm.csv": 69,
        }
        print("Data Variant 3: downsampling")

    elif DATA_VARIANT == 4:
        # exp026/s012 like — priority upweighting exp026/s011 case
        BASE_SAMPLES = {
            "numeral_system.csv": 600,
            "gravity_physics.csv": 1200,
            "unit_conversion.csv": 1150,
            "text_decryption.csv": 1492,
            "bit_manipulation_including_wrong.csv": 1508,
            "bit_manipulation_synth_including_wrong_v2.csv": 500,
            "equation_numeric.csv": 535,
            "cryptarithm.csv": 69,
        }
        # Load priority IDs from uploaded dataset
        PRIORITY_FILE = "/kaggle/input/datasets/konbu17/exp024-tong-style-cot-data/priority/exp026_s011_priority.txt"
        priority_ids = set()
        if os.path.exists(PRIORITY_FILE):
            with open(PRIORITY_FILE) as pf:
                priority_ids = {line.strip() for line in pf if line.strip()}
            print(f"Loaded {len(priority_ids)} priority problem IDs")
        else:
            print(f"WARNING: Priority file not found: {PRIORITY_FILE}")

        print("Data Variant 4: Priority upweighting (exp026/s011 style)")

        # Build records with ID tracking for priority duplication
        records = []
        id_records = []
        for fname, n in BASE_SAMPLES.items():
            path = os.path.join(TONG_DIR, fname)
            df = pd.read_csv(path)
            df = df[df["generated_cot"].notna() & (df["generated_cot"].str.len() > 5)]
            sampled = df.sample(n=min(n, len(df)), random_state=SEED)
            print(f"  {fname}: {len(sampled)}/{n}")
            for _, row in sampled.iterrows():
                rec = build_record(row["prompt"], row["answer"], row["generated_cot"])
                records.append(rec)
                id_records.append((str(row["id"]), rec))

        print(f"Base records: {len(records)}")

        # Add priority duplicates (hard problems get 2x training weight)
        dup_count = 0
        for pid, rec in list(id_records):
            if pid in priority_ids:
                records.append(dict(rec))
                dup_count += 1
        print(f"Priority duplicates: {dup_count}")
        print(f"Total records: {len(records)}")
    
    else:
        raise ValueError(f"Unknown DATA_VARIANT: {DATA_VARIANT}")

    def build_record(prompt, answer, cot):
        user = prompt + PROMPT_SUFFIX
        cot = str(cot).strip()
        answer = str(answer).strip()
        if f'\\boxed{{{answer}}}' in cot and '</think>' in cot:
            assistant = cot
        else:
            c = re.sub(r'\\boxed\{[^}]*\}', '', cot).strip()
            if '</think>' in c: c = c.split('</think>')[0].strip()
            assistant = f"{c}\n</think>\n\\boxed{{{answer}}}"
        return {"messages": [{"role": "user", "content": user}, {"role": "assistant", "content": assistant}]}

    records = []
    for fname, n in SAMPLES.items():
        path = os.path.join(TONG_DIR, fname)
        df = pd.read_csv(path)
        df = df[df["generated_cot"].notna() & (df["generated_cot"].str.len() > 5)]
        sampled = df.sample(n=min(n, len(df)), random_state=SEED)
        print(f"  {fname}: {len(sampled)}/{n}")
        for _, row in sampled.iterrows():
            records.append(build_record(row["prompt"], row["answer"], row["generated_cot"]))

    random.seed(SEED)
    random.shuffle(records)
    print(f"\nTotal training records: {len(records)}")

    train_dataset = HFDataset.from_list(records)

    def formatting_func(example):
        return tokenizer.apply_chat_template(
            example["messages"], tokenize=False,
            add_generation_prompt=False, enable_thinking=True)

    # --- 1st place style hyperparameters ---
    training_args = SFTConfig(
        output_dir="/kaggle/working/sft_adapter",
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=64,  # effective batch = 1*1*64 = 64 (single GPU)
        learning_rate=2e-4, # 2.4e-4
        lr_scheduler_type="linear",
        warmup_ratio=0.0,
        weight_decay=0.0,
        max_grad_norm=1e9,
        bf16=True,
        logging_steps=5,
        save_strategy="no",
        max_length=4096, # 8192 # 4096 # It should be 8192 but OOM. # Be carefull!!
        packing=False,
        seed=42,
        report_to="none",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        remove_unused_columns=False,
        adam_beta1=0.9,
        adam_beta2=0.95,
        adam_epsilon=1e-8,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        formatting_func=formatting_func,
        processing_class=tokenizer,
    )

    print(f"\nStarting training: {len(records)} samples, 1 epoch")
    trainer.train()

    # Save
    model.save_pretrained("/kaggle/working/sft_adapter")
    tokenizer.save_pretrained("/kaggle/working/sft_adapter")
    print("Training complete! Adapter saved.")

## Mode B: Load Pre-trained LoRA

In [ ]:
if USE_PRETRAINED:
    import os

    SRC_ADAPTER_DIR = PRETRAINED_ADAPTER_DATASET_PATH
    required_files = ["adapter_config.json", "adapter_model.safetensors"]

    print("Using pre-trained adapter from:", SRC_ADAPTER_DIR)
    for fname in required_files:
        fpath = os.path.join(SRC_ADAPTER_DIR, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing required adapter file: {fpath}")
        print(f"  {fname}: {os.path.getsize(fpath)/1024/1024:.1f} MB")
else:
    print("TRAIN_ON_KAGGLE=1: adapter already trained above")

## Create submission.zip

In [ ]:
import json, os, shutil, zipfile

OUTPUT_DIR = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]

if TRAIN_ON_KAGGLE:
    src_adapter_dir = "/kaggle/working/sft_adapter"
    print("Packaging freshly trained adapter from:", src_adapter_dir)
else:
    src_adapter_dir = PRETRAINED_ADAPTER_DATASET_PATH
    print("Packaging pre-trained adapter from:", src_adapter_dir)

for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1024 / 1024
        print(f"  Copied {fname}: {size_mb:.1f} MB")
    else:
        raise FileNotFoundError(f"Missing: {src}")

# Create submission.zip
zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)

zip_size = os.path.getsize(zip_path) / 1024 / 1024
print(f"\nsubmission.zip created: {zip_size:.1f} MB")
print("Done! submission.zip is ready for submission.")

In [ ]:
# how to get # /kaggle/input/datasets/konbu17/exp024-tong-style-cot-data/priority/exp026_s005_priority.txt
# priority file (duplication of low logprob data record)
"""
def run_logprob_analysis(llm, tokenizer):
    # Compute per-sample min logprob on training data using vLLM prompt_logprobs.
    # Uses TP=4 vLLM engine for fast batched processing.
    # Saves priority_problem_ids.txt and logprob_analysis.jsonl.
    
    from vllm import SamplingParams


    if not TRAIN_DATA.exists() or not TRAIN_META.exists():
        print("No training data/metadata found, skipping logprob analysis.")
        return


    print(f"\n{'='*60}\nLogprob Analysis (vLLM prompt_logprobs)\n{'='*60}")


    # Load training data and metadata
    train_ds = [json.loads(l) for l in open(TRAIN_DATA) if l.strip()]
    metadata = [json.loads(l) for l in open(TRAIN_META) if l.strip()]
    print(f"Training samples: {len(train_ds)}, Metadata: {len(metadata)}")


    # Prepare full conversation texts and prompt lengths
    full_texts = []
    prompt_lens = []
    for example in train_ds:
        full_text = tokenizer.apply_chat_template(
            example["messages"], tokenize=False,
            add_generation_prompt=False, enable_thinking=True)
        prompt_text = tokenizer.apply_chat_template(
            example["messages"][:1], tokenize=False,
            add_generation_prompt=True, enable_thinking=True)
        prompt_len = len(tokenizer.encode(prompt_text, add_special_tokens=False))
        full_texts.append(full_text)
        prompt_lens.append(prompt_len)


    # Tokenize full texts to get token IDs (needed to look up logprobs)
    all_token_ids = []
    for text in full_texts:
        ids = tokenizer.encode(text, add_special_tokens=False)
        all_token_ids.append(ids)


    # Run vLLM with prompt_logprobs — process in chunks to manage memory
    CHUNK_SIZE = 500
    all_results = []
    n_chunks = (len(full_texts) + CHUNK_SIZE - 1) // CHUNK_SIZE


    for chunk_idx in range(n_chunks):
        start = chunk_idx * CHUNK_SIZE
        end = min(start + CHUNK_SIZE, len(full_texts))
        chunk_texts = full_texts[start:end]


        outputs = llm.generate(
            chunk_texts,
            SamplingParams(prompt_logprobs=1, max_tokens=1, temperature=0.0))


        for j, output in enumerate(outputs):
            i = start + j
            plps = output.prompt_logprobs
            prompt_len = prompt_lens[i]
            token_ids = all_token_ids[i]


            # Extract completion logprobs (tokens after prompt)
            completion_logprobs = []
            for pos in range(prompt_len, min(len(plps), len(token_ids))):
                if plps[pos] is None:
                    continue
                tid = token_ids[pos]
                if tid in plps[pos]:
                    completion_logprobs.append(plps[pos][tid].logprob)


            if not completion_logprobs:
                continue


            min_lp = min(completion_logprobs)
            mean_lp = sum(completion_logprobs) / len(completion_logprobs)
            meta = metadata[i] if i < len(metadata) else {"problem_id": f"unk_{i}", "category": "unknown"}
            all_results.append({
                "problem_id": meta["problem_id"],
                "category": meta["category"],
                "min_logprob": min_lp,
                "mean_logprob": mean_lp,
                "n_tokens": len(completion_logprobs),
            })


        print(f"  Chunk {chunk_idx+1}/{n_chunks}: processed {end}/{len(full_texts)}")


    # Summary by category
    cat_results = defaultdict(list)
    for r in all_results:
        cat_results[r["category"]].append(r["min_logprob"])


    print(f"\n=== Min Logprob by Category ===")
    for cat in sorted(cat_results.keys()):
        lps = cat_results[cat]
        n_below = sum(1 for lp in lps if lp < LOGPROB_THRESHOLD)
        print(f"  {cat}: min={min(lps):.3f}, mean={sum(lps)/len(lps):.3f}, "
              f"below_{LOGPROB_THRESHOLD:.2f}={n_below}/{len(lps)}")


    # Save priority IDs
    priority_ids = set()
    for r in all_results:
        pid = r["problem_id"]
        if r["min_logprob"] < LOGPROB_THRESHOLD and "-p0" not in pid and "-d" not in pid:
            priority_ids.add(pid)


    priority_path = EXP_DIR / "priority_problem_ids.txt"
    with open(priority_path, 'w') as f:
        for pid in sorted(priority_ids):
            f.write(pid + '\n')
    print(f"\nPriority problems (min_lp < {LOGPROB_THRESHOLD}): {len(priority_ids)}")
    print(f"Saved: {priority_path}")


    # Save full results
    results_path = EXP_DIR / "logprob_analysis.jsonl"
    with open(results_path, 'w') as f:
        for r in sorted(all_results, key=lambda x: x["min_logprob"]):
            f.write(json.dumps(r) + '\n')
    print(f"Saved: {results_path}")
"""